# Exercise 3: RAG & Knowledge Integration

## Learning Objectives

In this exercise, you will:
- Understand the difference between tools (function calling) and RAG (knowledge retrieval)
- Configure and use Vertex AI RAG Engine for destination knowledge
- Create RAG-only agents with knowledge base access
- Test semantic retrieval with destination guides
- Combine function calling tools with RAG in a hybrid agent pattern

**Estimated time:** 40 minutes

## What is RAG (Retrieval-Augmented Generation)?

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: This builds on Phase 2 - emphasize the complementary relationship -->

RAG gives your agent access to **private knowledge** - documents, guides, policies - that weren't in the LLM's training data. The agent retrieves relevant information from a knowledge base and uses it to generate accurate, grounded responses.

---

### How RAG Works

```
1. Documents indexed into RAG corpus (PDFs → chunks → embeddings)
          ↓
2. User asks: "What are visa requirements for Japan?"
          ↓
3. Agent searches corpus (semantic similarity)
          ↓
4. Top relevant chunks retrieved (with similarity scores)
          ↓
5. Agent uses chunks as context to answer question
```

---

### Tools vs RAG: The Decision Framework (Revisited)

Recall from Exercise 2 - you choose based on data characteristics:

```mermaid
flowchart TD
    A[User Query] --> B{Does this need\nREAL-TIME data?}
    B -->|YES| C[Use FUNCTION CALLING\nsearch_flights, search_hotels]
    B -->|NO| D{Is this STATIC\nKNOWLEDGE?}
    D -->|YES| E[Use RAG RETRIEVAL\nretrieve_destination_info]
    D -->|NO| F[Direct LLM\ngeneral knowledge]
```

| Question | Tool or RAG? | Why? |
|----------|--------------|------|
| "Find flights to Tokyo" | **TOOL** (search_flights) | Prices/availability change constantly |
| "What's the best time to visit Tokyo?" | **RAG** (retrieve_destination_info) | Static seasonal guide |
| "Book hotel in Shibuya under $200" | **TOOL** (search_hotels) | Real-time pricing |
| "What are visa requirements for Japan?" | **RAG** (retrieve_destination_info) | Static immigration policy |
| "What's the capital of Japan?" | **Neither** (LLM knows this) | General knowledge |

> 💡 **Key Insight:** Use **tools for dynamic data** that changes while user is talking. Use **RAG for static knowledge** in documents.

---

### Why RAG for Travel Guides?

**Perfect use case:**
- Destination guides rarely change (update quarterly, not hourly)
- Contains structured information (visa rules, attractions, cultural tips)
- Too much content to fit in LLM context window
- Needs semantic search ("safety tips" retrieves safety sections)

**In this workshop:**
- Pre-indexed corpus: 10 destination guides (Tokyo, Paris, NYC, Singapore, etc.)
- Each guide: visa requirements, attractions, weather, culture, transportation, food
- Layout-aware chunking: tables and lists preserved intact
- Document AI parser: understands document structure for better retrieval

## Setup Instructions

**⏱️ Estimated time: ~2 minutes**

This exercise assumes you have completed Exercise 1 and Exercise 2.

If you haven't completed setup yet, please go back to: [00-setup-verification.ipynb](00-setup-verification.ipynb)

In [ ]:
# @title Quick Setup (Run this cell first)
# This cell sets up your environment - you can collapse it after running

import os
import google.generativeai as genai
from getpass import getpass

# Step 1: Install Google ADK
!pip install -q google-adk==1.23.0
print("✓ google-adk 1.23.0 installed")

# Step 2: Configure API Key
api_key = getpass('Enter your Google AI API Key: ')
genai.configure(api_key=api_key)
os.environ['GOOGLE_API_KEY'] = api_key
print("✓ API Key configured")

print("\n🚀 Ready to integrate RAG knowledge!")

In [ ]:
# Import ADK components for agents and RAG
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part
from google.adk.tools.retrieval.vertex_ai_rag_retrieval import VertexAiRagRetrieval
from vertexai.preview import rag

# Create session service for conversation management
session_service = InMemorySessionService()
user_id = "workshop_participant"

print("✓ ADK components imported")
print("✓ RAG retrieval tools imported")
print("✓ Session service created")

## Exercise 3A: Explore Pre-Indexed RAG Corpus

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: Show the corpus in the console if possible -->

The instructor has pre-created a RAG corpus with destination guide PDFs. This saves 15-20 minutes of setup time and lets us focus on **using** RAG, not infrastructure.

### What's in the Corpus?

**10 destination guides**, each with standardized sections:
1. Quick Facts (language, currency, emergency numbers)
2. Visa Requirements (by nationality)
3. Best Time to Visit (seasonal breakdown)
4. Top Attractions (table format with fees, hours, tips)
5. Neighborhoods & Districts
6. Food & Dining
7. Transportation
8. Cultural Tips & Customs
9. Safety & Health
10. Practical Information

**Destinations covered:**
- Tokyo, Japan
- Paris, France
- New York City, USA
- Singapore
- London, UK
- Rome, Italy
- Bangkok, Thailand
- Sydney, Australia
- Barcelona, Spain
- Dubai, UAE

### How the Guides Were Indexed

**Chunking strategy:**
- Chunk size: 1024 tokens (~800 words)
- Chunk overlap: 256 tokens (~200 words)
- Document AI layout parser enabled (preserves tables/lists)

**Why layout parsing matters:**
- WITHOUT: Table splits mid-row → broken data
- WITH: Full tables in single chunk → clean retrieval

Example:
```
| Attraction | Price | Hours |
|------------|-------|-------|
| Temple     | Free  | 6am   |
| Museum     | $15   | 9am   |
```
↑ This stays intact in one chunk with layout parser

In [ ]:
# Configure RAG Corpus ID
# The instructor will provide this during the workshop

# TODO: Get this from your instructor
# Format: projects/{project}/locations/{location}/ragCorpora/{corpus_id}
RAG_CORPUS_ID = "projects/your-workshop-project/locations/us-central1/ragCorpora/1234567890"

# Store in environment for later use
os.environ['RAG_CORPUS_ID'] = RAG_CORPUS_ID

print(f"✓ RAG Corpus ID configured")
print(f"  Corpus: {RAG_CORPUS_ID[:60]}...")
print("\n📚 Corpus contains 10 destination guides with ~15-25 chunks each")

### ✅ Checkpoint: Corpus Configured

<!-- INSTRUCTOR: Verify all participants have the corpus ID before proceeding -->

You should see:
- ✓ RAG Corpus ID configured
- Corpus path starting with `projects/...`

**If you see an error:**
- Make sure you replaced the placeholder corpus ID with the one from your instructor
- Check that the format matches: `projects/{project}/locations/{location}/ragCorpora/{id}`

---

## ✏️ Exercise 3B: Configure RAG Retrieval Tool

**⏱️ Estimated time: ~7 minutes**

<!-- INSTRUCTOR: Emphasize the DO/DO NOT pattern - critical for tool selection -->

Now let's create a RAG retrieval tool that searches the destination guide corpus.

### VertexAiRagRetrieval Parameters

| Parameter | Type | Purpose | Recommended Value |
|-----------|------|---------|-------------------|
| `name` | str | Function name LLM calls | `retrieve_destination_info` |
| `description` | str | **Critical!** Tells LLM when to use this tool | See pattern below |
| `rag_resources` | list | Corpus reference(s) to search | `[rag.RagResource(rag_corpus=...)]` |
| `similarity_top_k` | int | How many chunks to retrieve | `5` (default) |
| `vector_distance_threshold` | float | Minimum similarity score (0-1) | `0.6` (filters low quality) |

### The DO/DO NOT Description Pattern

**Why this matters:** The LLM reads your tool description to decide when to call it. Vague descriptions cause tool confusion!

❌ **Bad description:**
```python
description='Get destination information'  # Too vague!
```
Result: LLM calls this for "Find flights to Tokyo" 😞

✅ **Good description:**
```python
description='''
Retrieve destination information from travel guide knowledge base.

USE THIS TOOL to answer questions about:
- Visa requirements and entry rules
- Top attractions and landmarks  
- Weather and best time to visit
- Cultural tips and customs

DO NOT use this tool for:
- Real-time flight availability → use search_flights() instead
- Real-time hotel availability → use search_hotels() instead
'''
```
Result: LLM calls RAG for guides, tools for bookings 🎯

### Your Task

Complete the `VertexAiRagRetrieval` configuration below:

In [ ]:
# Exercise 3B: Configure RAG Tool

destination_knowledge = VertexAiRagRetrieval(
    name='retrieve_destination_info',
    
    # TODO: Write explicit description with DO/DO NOT sections
    # Hint: Explain what information this tool provides
    # Hint: Clarify when to use this vs booking tools (search_flights, search_hotels)
    description='''
    TODO: Fill in description
    
    USE THIS TOOL to answer questions about:
    - TODO: List what RAG provides (visa, attractions, culture...)
    
    DO NOT use this tool for:
    - TODO: List what tools provide (real-time flights, hotels...)
    ''',
    
    # TODO: Configure RAG resources with corpus ID
    rag_resources=[
        rag.RagResource(
            rag_corpus=os.environ['RAG_CORPUS_ID']  # Use the environment variable
        )
    ],
    
    # TODO: Set retrieval parameters
    similarity_top_k=5,  # TODO: How many chunks to retrieve? (try 5)
    vector_distance_threshold=0.6,  # TODO: Minimum similarity? (try 0.6)
)

print("✓ RAG tool configured")
print(f"  Retrieval: top {destination_knowledge.similarity_top_k} chunks")
print(f"  Threshold: {destination_knowledge.vector_distance_threshold} similarity")

### 💡 Solution: Complete Configuration

Click to expand if you need help:

In [ ]:
# @title Solution: Exercise 3B (Expand to see)

destination_knowledge_solution = VertexAiRagRetrieval(
    name='retrieve_destination_info',
    
    description='''Retrieve destination information from travel guide knowledge base.

USE THIS TOOL to answer questions about:
- Visa requirements and entry rules (static immigration policy)
- Top attractions and landmarks (guide recommendations)
- Best time to visit by season (weather patterns, events)
- Cultural tips and local customs (etiquette, traditions)
- Safety information and travel advisories
- Transportation within the city (metro, buses, taxis)
- Food and dining recommendations (local cuisine, prices)
- Neighborhoods and districts to explore
- Practical travel information (plugs, currency, language)

DO NOT use this tool for:
- Real-time flight availability → use search_flights() instead
- Real-time hotel availability → use search_hotels() instead
- Current pricing or booking status → use search tools
- Live event schedules or ticket availability

This tool searches STATIC destination guides, not live databases.
Use for travel planning knowledge, not booking transactions.''',
    
    rag_resources=[
        rag.RagResource(
            rag_corpus=os.environ['RAG_CORPUS_ID']
        )
    ],
    
    similarity_top_k=5,
    vector_distance_threshold=0.6,
)

print("✓ Solution RAG tool configured")

### ✅ Checkpoint: Tool Configured

You should see:
- ✓ RAG tool configured
- Retrieval: top 5 chunks
- Threshold: 0.6 similarity

**If you get an error:**
- Check that `RAG_CORPUS_ID` is set (from Exercise 3A)
- Verify the description is a multi-line string (use triple quotes `'''`)
- Make sure all parameters are filled in

---

## ✏️ Exercise 3C: Create RAG-Only Agent

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: Explain the single-tool constraint clearly -->

Now create an agent that uses **ONLY** the RAG retrieval tool for destination knowledge.

### ⚠️ Important ADK Constraint

**VertexAiRagRetrieval can ONLY be used by itself** - you cannot combine it with function calling tools in the same agent.

❌ **This will NOT work:**
```python
agent = Agent(
    tools=[
        search_flights,        # Function calling tool
        search_hotels,         # Function calling tool  
        destination_knowledge, # RAG tool ← VIOLATION!
    ]
)
```

✅ **This WILL work:**
```python
# RAG-only agent
destination_agent = Agent(
    tools=[destination_knowledge]  # ONLY RAG tool
)
```

**Why this constraint?** ADK's architecture limitation. We'll solve this in Exercise 3E with a "hybrid agent" pattern.

### Your Task

Complete the agent configuration:

In [ ]:
# Exercise 3C: Create RAG-Only Agent

destination_expert = Agent(
    model='gemini-2.5-flash',
    name='destination_expert',
    
    # TODO: Write agent description
    description='TODO: Describe this agent (e.g., "Travel destination expert with knowledge base access")',
    
    # TODO: Write agent instruction
    instruction='''TODO: Write instruction for the agent
    
    Your instruction should explain:
    - What the agent can do (retrieve from destination guides)
    - What it cannot do (real-time booking)
    - How to use the retrieve_destination_info tool
    - When to cite information from guides
    
    Example structure:
    "You are a travel destination expert with access to comprehensive guides.
    
    YOUR CAPABILITIES:
    - Retrieve information from destination guides using your tool
    - Answer questions about visa requirements, attractions, weather, culture
    ...
    
    YOUR LIMITATIONS:
    - You CANNOT search for flights or hotels (no real-time booking data)
    ...
    
    HOW TO HELP:
    1. Use retrieve_destination_info tool for all destination queries
    2. Combine information from multiple guide sections
    3. Cite specific details when available
    ..."
    ''',
    
    # TODO: Add ONLY the RAG tool (single-tool constraint!)
    tools=[destination_knowledge],  # ONLY this tool, no others!
)

print("✓ Destination expert agent created")
print(f"  Model: {destination_expert.model}")
print(f"  Tools: {len(destination_expert.tools)} (RAG only)")

### 💡 Solution: Complete Agent Configuration

Click to expand if you need help:

In [ ]:
# @title Solution: Exercise 3C (Expand to see)

destination_expert_solution = Agent(
    model='gemini-2.5-flash',
    name='destination_expert',
    description='Travel destination expert with access to comprehensive destination guides.',
    
    instruction='''You are a travel destination expert with access to comprehensive destination guides.

YOUR CAPABILITIES:
- Retrieve information from destination guides using your tool
- Answer questions about visa requirements, attractions, weather, culture
- Provide cultural tips, safety info, transportation advice
- Share food and dining recommendations

YOUR LIMITATIONS:
- You CANNOT search for flights or hotels (no real-time booking data)
- You CANNOT provide current prices or booking availability
- Your knowledge comes from static travel guides

HOW TO HELP:
1. Use retrieve_destination_info tool for all destination queries
2. Combine information from multiple guide sections for comprehensive answers
3. Cite specific details from guides when available
4. If guide doesn't cover a topic, admit this honestly
5. Recommend using booking tools for real-time availability

Be informative, encouraging, and help travelers prepare confidently.''',
    
    tools=[destination_knowledge_solution],
)

print("✓ Solution destination expert created")

### ✅ Checkpoint: Agent Created

You should see:
- ✓ Destination expert agent created
- Model: gemini-2.5-flash
- Tools: 1 (RAG only)

**Common issues:**
- If "Tools: 0" → You forgot to add `destination_knowledge` to the tools list
- If "Tools: 2+" → You added other tools - remove them! RAG must be alone